<a href="https://colab.research.google.com/github/Elgendi/The-Effect-Separation-Index/blob/main/ESI_complete_Colab_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Effect Separation Index — complete corrected Colab analysis

This notebook reproduces the final corrected analysis, all main and supplementary figures, all tables and CSV source data, and the formatted journal Excel workbook.

**Expected runtime:** approximately 10–20 minutes on a standard Colab CPU. Do not interrupt the 2,000-bootstrap real-data stage. Run every cell in order.


In [ ]:
!pip -q install numpy pandas scipy scikit-learn matplotlib xlsxwriter openpyxl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.4 MB/s eta 0:00:00


## 1. Write the canonical corrected analysis pipeline


In [ ]:
%%writefile esi_pipeline.py
"""Canonical reproducible analysis for the Effect Separation Index manuscript.

Run: python esi_pipeline.py
Outputs are written to outputs/esi_submission/.
"""
from __future__ import annotations

import hashlib, io, json, platform, sys, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
from scipy.integrate import trapezoid
from scipy.ndimage import gaussian_filter1d
from scipy.stats import boxcox, norm, rankdata, ttest_ind
from sklearn.datasets import load_breast_cancer
from sklearn.neighbors import KernelDensity
import sklearn
import matplotlib as mpl
import matplotlib.pyplot as plt

MASTER_SEED = 2025
N_BOOTSTRAPS = 2000
N_MONTE_CARLO = 500
KDE_GRID_SIZE = 512
MIN_VALID_FRACTION = 0.95
OUT = Path("outputs/esi_submission")
DATA = OUT / "downloaded_data"
FIG = OUT / "figures"
CSV = OUT / "source_csv"
EPS = np.finfo(float).eps

URLS = {
 "Pima Diabetes": "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv",
 "Cleveland Heart Disease": "https://raw.githubusercontent.com/amankharwal/Website-data/master/heart.csv",
 "Framingham CHD": "https://raw.githubusercontent.com/kabdi101/CHD_prediction/master/framingham.csv",
 "Fetal Health": "https://raw.githubusercontent.com/rrahulg/fetal-health/main/fetal_health.csv",
 "Heart Disease Prediction": "https://raw.githubusercontent.com/GuySuphakit/Heart-Failure-Prediction/main/heart.csv",
 "Haberman Survival": "https://raw.githubusercontent.com/jbrownlee/Datasets/master/haberman.csv",
 "Heart Failure Clinical Records": "https://archive.ics.uci.edu/static/public/519/data.csv",
}

PRIMARY_VARIABLES = {
 "Breast Cancer Wisconsin": ["worst area", "worst concavity", "mean area", "mean texture", "mean radius"],
 "Pima Diabetes": ["Glucose", "BMI", "BloodPressure", "Age", "DiabetesPedigreeFunction", "Insulin"],
 "Cleveland Heart Disease": ["chol", "oldpeak", "thalach", "age", "trestbps"],
 "Framingham CHD": ["totChol", "sysBP", "age", "BMI", "heartRate", "diaBP", "glucose"],
 "Fetal Health": ["fetal_movement", "histogram_max", "accelerations", "uterine_contractions", "mean_value_of_short_term_variability", "baseline value", "histogram_variance"],
 "Heart Disease Prediction": ["RestingBP", "Age", "MaxHR", "Oldpeak", "Cholesterol"],
 "Haberman Survival": ["PositiveNodes", "Age", "OperationYear"],
 "Heart Failure Clinical Records": ["age", "ejection_fraction", "serum_creatinine", "serum_sodium", "platelets"],
}

def download(name: str, *, header="infer", names=None) -> tuple[pd.DataFrame, dict]:
    url = URLS[name]
    payload = urllib.request.urlopen(url, timeout=60).read()
    path = DATA / (name.lower().replace(" ", "_") + ".csv")
    path.write_bytes(payload)
    frame = pd.read_csv(io.BytesIO(payload), header=header, names=names)
    return frame, {"Dataset": name, "Source_URL": url, "Downloaded_file": path.name,
                   "SHA256": hashlib.sha256(payload).hexdigest(), "Rows_downloaded": len(frame),
                   "Columns_downloaded": frame.shape[1], "Access_date": "2026-08-31"}

def pooled_transform(x, y, method):
    x, y = np.asarray(x,float), np.asarray(y,float)
    z = np.r_[x,y]
    if method == "Raw": t=z
    elif method == "Pooled z-score": t=(z-z.mean())/z.std(ddof=1)
    elif method == "Pooled min-max": t=(z-z.min())/np.ptp(z) if np.ptp(z)>EPS else np.zeros_like(z)
    elif method.startswith("Log"):
        shifted=z-z.min()+max(EPS, np.ptp(z)*1e-9); t=np.log(shifted)
        if method.endswith("z-score"): t=(t-t.mean())/t.std(ddof=1)
        else: t=(t-t.min())/np.ptp(t)
    elif method.startswith("Box-Cox"):
        shifted=z-z.min()+max(EPS,np.ptp(z)*1e-9); t,_=boxcox(shifted)
        if method.endswith("z-score"): t=(t-t.mean())/t.std(ddof=1)
        else: t=(t-t.min())/np.ptp(t)
    elif method == "Rank-normal + min-max":
        t=norm.ppf((rankdata(z,method="average")-.5)/len(z)); t=(t-t.min())/np.ptp(t)
    else: raise ValueError(method)
    return t[:len(x)],t[len(x):]

TRANSFORMS=["Raw","Pooled z-score","Pooled min-max","Log + z-score","Log + min-max",
            "Box-Cox + z-score","Box-Cox + min-max","Rank-normal + min-max"]

def cohens_d(x,y):
    nx,ny=len(x),len(y); v=((nx-1)*np.var(x,ddof=1)+(ny-1)*np.var(y,ddof=1))/(nx+ny-2)
    return np.nan if v<=EPS else float((np.mean(y)-np.mean(x))/np.sqrt(v))

def bandwidth(a):
    sd=np.std(a,ddof=1); iq=(np.percentile(a,75)-np.percentile(a,25))/1.349
    vals=[v for v in (sd,iq) if np.isfinite(v) and v>EPS]
    return np.nan if not vals else .9*min(vals)*len(a)**(-.2)

def overlap(x,y,bw_mult=1.0):
    hx,hy=bandwidth(x)*bw_mult,bandwidth(y)*bw_mult
    if not np.isfinite(hx+hy): return np.nan
    z=np.r_[x,y]; ext=4*max(hx,hy); edges=np.linspace(z.min()-ext,z.max()+ext,KDE_GRID_SIZE+1)
    ctr=(edges[:-1]+edges[1:])/2; step=edges[1]-edges[0]
    cx,_=np.histogram(x,bins=edges); cy,_=np.histogram(y,bins=edges)
    fx=gaussian_filter1d(cx/(len(x)*step),hx/step,mode="constant")
    fy=gaussian_filter1d(cy/(len(y)*step),hy/step,mode="constant")
    return float(np.clip(trapezoid(np.minimum(fx,fy),x=ctr),0,1))

def overlap_kernel(x, y, kernel="gaussian", bw_mult=1.0):
    """KDE overlap for the six reviewer-requested sklearn kernels."""
    hx,hy=bandwidth(x)*bw_mult,bandwidth(y)*bw_mult
    if not np.isfinite(hx+hy): return np.nan
    z=np.r_[x,y]; ext=4*max(hx,hy); grid=np.linspace(z.min()-ext,z.max()+ext,KDE_GRID_SIZE)[:,None]
    fx=np.exp(KernelDensity(kernel=kernel,bandwidth=hx).fit(np.asarray(x)[:,None]).score_samples(grid))
    fy=np.exp(KernelDensity(kernel=kernel,bandwidth=hy).fit(np.asarray(y)[:,None]).score_samples(grid))
    return float(np.clip(trapezoid(np.minimum(fx,fy),x=grid[:,0]),0,1))

def metrics(x,y,seed,n_boot=N_BOOTSTRAPS,bw_mult=1.0):
    x,y=np.asarray(x,float),np.asarray(y,float); rng=np.random.default_rng(seed)
    ix=rng.integers(0,len(x),(n_boot,len(x))); iy=rng.integers(0,len(y),(n_boot,len(y)))
    ds=np.empty(n_boot); ovs=np.empty(n_boot)
    for b in range(n_boot): ds[b]=cohens_d(x[ix[b]],y[iy[b]]); ovs[b]=overlap(x[ix[b]],y[iy[b]],bw_mult)
    good=np.isfinite(ds)&np.isfinite(ovs)
    if good.sum()<MIN_VALID_FRACTION*n_boot: raise RuntimeError("Too few valid bootstraps")
    ds,ovs=ds[good],ovs[good]; med=float(np.median(np.abs(ds))); sd=float(np.std(ds,ddof=1)); ov=float(np.mean(ovs))
    return {"P_value":float(ttest_ind(x,y,equal_var=False).pvalue),"Cohens_d":cohens_d(x,y),
            "Median_abs_d_boot":med,"SD_d_boot":sd,"Mean_OVL_boot":ov,
            "ESI":med/sd*(1-ov) if sd>EPS else np.nan,"Valid_bootstraps":len(ds)}

def load_datasets():
    manifest=[]; specs=[]; overview=[]
    bc=load_breast_cancer(as_frame=True).frame.copy(); bc_meta={"Dataset":"Breast Cancer Wisconsin","Source_URL":"https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html","Downloaded_file":"bundled_with_sklearn","SHA256":"version-controlled sklearn bundle","Rows_downloaded":len(bc),"Columns_downloaded":bc.shape[1],"Access_date":"2026-08-31"}; manifest.append(bc_meta)
    datasets=[("Breast Cancer Wisconsin",bc,"target",0,1,"malignant","benign","No missing values; sklearn target 0=malignant, 1=benign")]
    cols=["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin","BMI","DiabetesPedigreeFunction","Age","Outcome"]
    p,m=download("Pima Diabetes",header=None,names=cols); manifest.append(m)
    for v in ["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]: p.loc[p[v]==0,v]=np.nan
    datasets.append(("Pima Diabetes",p,"Outcome",0,1,"no diabetes","diabetes","Physiologically impossible zeros set to missing per variable; complete-case per comparison"))
    h,m=download("Cleveland Heart Disease"); manifest.append(m); h["target_binary"]=(pd.to_numeric(h["target"])>0).astype(int)
    datasets.append(("Cleveland Heart Disease",h,"target_binary",0,1,"no disease","disease","Complete-case per comparison; target >0 collapsed to disease"))
    f,m=download("Framingham CHD"); manifest.append(m); datasets.append(("Framingham CHD",f,"TenYearCHD",0,1,"no 10-year CHD","10-year CHD","Complete-case per comparison"))
    ft,m=download("Fetal Health"); manifest.append(m); ft=ft[ft.fetal_health.isin([1.,3.])].copy()
    datasets.append(("Fetal Health",ft,"fetal_health",1.,3.,"normal","pathological","Suspect class excluded; complete-case per comparison"))
    hp,m=download("Heart Disease Prediction"); manifest.append(m); datasets.append(("Heart Disease Prediction",hp,"HeartDisease",0,1,"no disease","disease","Complete-case per comparison"))
    ha,m=download("Haberman Survival",header=None); manifest.append(m); ha.columns=["Age","OperationYear","PositiveNodes","SurvivalStatus"]
    datasets.append(("Haberman Survival",ha,"SurvivalStatus",1,2,"survived >=5 years","died within 5 years","Complete-case per comparison"))
    hf,m=download("Heart Failure Clinical Records"); manifest.append(m); datasets.append(("Heart Failure Clinical Records",hf,"death_event",0,1,"survived follow-up","died during follow-up","Complete-case per comparison"))
    for di,(name,df,gcol,ga,gb,la,lb,missing) in enumerate(datasets):
        for vi,var in enumerate(PRIMARY_VARIABLES[name]): specs.append((di,vi,name,df,var,gcol,ga,gb,la,lb))
        valid_group=df[gcol].isin([ga,gb]); overview.append({"Dataset":name,"Total_N_used":int(valid_group.sum()),"Group_A":la,"Group_A_N":int((df[gcol]==ga).sum()),"Group_B":lb,"Group_B_N":int((df[gcol]==gb).sum()),"Outcome_definition":gcol,"Variables_analysed":", ".join(PRIMARY_VARIABLES[name]),"Missing_data_handling":missing,"Source_URL":manifest[-1]["Source_URL"] if name=="Heart Failure Clinical Records" else next(z["Source_URL"] for z in manifest if z["Dataset"]==name)})
    return specs,pd.DataFrame(overview),pd.DataFrame(manifest)

def real_analysis(specs):
    rows=[]
    for di,vi,name,df,var,gcol,ga,gb,la,lb in specs:
        sub=df[[var,gcol]].copy(); sub[var]=pd.to_numeric(sub[var],errors="coerce"); sub=sub.dropna()
        x=sub.loc[sub[gcol]==ga,var].to_numpy(); y=sub.loc[sub[gcol]==gb,var].to_numpy(); seed=MASTER_SEED+(di+1)*10000+vi
        common={"Dataset":name,"Variable":var,"Group_A":la,"Group_B":lb,"N_A":len(x),"N_B":len(y)}
        raw=metrics(x,y,seed); tx,ty=pooled_transform(x,y,"Rank-normal + min-max"); tr=metrics(tx,ty,seed)
        for scale,res in [("Raw",raw),("Rank-normal + min-max",tr)]: rows.append(common|{"Scale":scale}|res)
    full=pd.DataFrame(rows); wide=full.pivot(index=["Dataset","Variable","Group_A","Group_B","N_A","N_B"],columns="Scale",values=["P_value","Cohens_d","Mean_OVL_boot","ESI"]).reset_index()
    wide.columns=["_".join([str(x) for x in c if x]) for c in wide.columns]; wide["Delta_ESI"]=wide["ESI_Rank-normal + min-max"]-wide["ESI_Raw"]
    return full,wide

def scenarios(rng,n,name):
    if name=="Normal clear": return rng.normal(0,1,n),rng.normal(1.4,1,n)
    if name=="Normal overlap": return rng.normal(0,1,n),rng.normal(.2,1,n)
    if name=="Skewed lognormal": return rng.lognormal(0,.7,n),rng.lognormal(.3,.7,n)
    return rng.exponential(1,n),rng.gamma(2,.5,n)

def simulation_analysis():
    rows=[]; ns=[10,20,50,100,200,500,2000]; names=["Normal clear","Normal overlap","Skewed lognormal","Exponential vs gamma"]
    for si,name in enumerate(names):
      for n in ns:
       for r in range(N_MONTE_CARLO):
        x,y=scenarios(np.random.default_rng(MASTER_SEED+si*100000+n*1000+r),n,name)
        for method in TRANSFORMS:
         tx,ty=pooled_transform(x,y,method); d=cohens_d(tx,ty); rows.append({"Scenario":name,"N_per_group":n,"Replicate":r,"Transformation":method,"P_value":ttest_ind(tx,ty,equal_var=False).pvalue,"Cohens_d":d,"Abs_d":abs(d),"OVL":overlap(tx,ty)})
    rep=pd.DataFrame(rows); summary=[]
    for keys,g in rep.groupby(["Scenario","N_per_group","Transformation"],sort=False):
      sd=g.Cohens_d.std(ddof=1); esi=g.Abs_d.median()/sd*(1-g.OVL.mean()) if sd>EPS else np.nan
      summary.append(dict(zip(["Scenario","N_per_group","Transformation"],keys))|{"Median_P":g.P_value.median(),"Median_abs_d":g.Abs_d.median(),"SD_d":sd,"Mean_OVL":g.OVL.mean(),"ESI":esi})
    return rep,pd.DataFrame(summary)

def sensitivity_examples(specs):
    # Exact requested use cases: a benefit candidate and the strongest non-benefit candidate are selected transparently later.
    kernel=[]; stability=[]
    x=np.random.default_rng(MASTER_SEED).lognormal(0,.7,200); y=np.random.default_rng(MASTER_SEED+1).lognormal(.3,.7,200)
    rngk=np.random.default_rng(MASTER_SEED+99); ixk=rngk.integers(0,len(x),(500,len(x))); iyk=rngk.integers(0,len(y),(500,len(y)))
    for kernel_name in ["gaussian","tophat","epanechnikov","exponential","linear","cosine"]:
     for mult in [.5,.75,1,1.25,1.5,2]:
      ds=[]; ovs=[]
      for b in range(500):
       xb,yb=x[ixk[b]],y[iyk[b]]; ds.append(cohens_d(xb,yb)); ovs.append(overlap_kernel(xb,yb,kernel_name,mult))
      ds=np.asarray(ds); ovs=np.asarray(ovs); good=np.isfinite(ds)&np.isfinite(ovs); med=np.median(np.abs(ds[good])); sd=np.std(ds[good],ddof=1)
      kernel.append({"Kernel":kernel_name,"Bandwidth_multiplier":mult,"Median_abs_d_boot":med,"SD_d_boot":sd,"Mean_OVL_boot":np.mean(ovs[good]),"ESI":med/sd*(1-np.mean(ovs[good])),"Valid_bootstraps":int(good.sum())})
    rng=np.random.default_rng(MASTER_SEED); ix=rng.integers(0,len(x),(N_BOOTSTRAPS,len(x))); iy=rng.integers(0,len(y),(N_BOOTSTRAPS,len(y)))
    ds=np.array([cohens_d(x[ix[b]],y[iy[b]]) for b in range(N_BOOTSTRAPS)]); med=np.median(np.abs(ds)); ov=overlap(x,y)
    denoms={"SD":np.std(ds,ddof=1),"Scaled MAD":1.4826*np.median(np.abs(ds-np.median(ds))),"Scaled IQR":(np.percentile(ds,75)-np.percentile(ds,25))/1.349,"95% CI width / 3.92":(np.percentile(ds,97.5)-np.percentile(ds,2.5))/3.92}
    for method,dn in denoms.items(): stability.append({"Stability_estimator":method,"Denominator":dn,"ESI":med/dn*(1-ov)})
    return pd.DataFrame(kernel),pd.DataFrame(stability)

def make_figures(sim,wide,full,kernel,stability):
    mpl.rcParams.update({"font.size":9,"axes.spines.top":False,"axes.spines.right":False,"figure.dpi":160})
    # Main Figure 1: corrected pooled transformation delta ESI at n=200.
    s=sim[sim.N_per_group==200]; raw=s[s.Transformation=="Raw"][["Scenario","ESI"]].rename(columns={"ESI":"Raw_ESI"}); d=s.merge(raw,on="Scenario"); d["Delta_ESI"]=d.ESI-d.Raw_ESI
    fig,axs=plt.subplots(2,2,figsize=(10,7),sharey=True)
    for ax,(name,g) in zip(axs.flat,d.groupby("Scenario",sort=False)):
      ax.barh(g.Transformation,g.Delta_ESI,color=["#9E9E9E" if t=="Raw" else "#0072B2" for t in g.Transformation]); ax.axvline(0,color="black",lw=.8); ax.set_title(name); ax.set_xlabel(r"$\Delta$ESI from raw")
    fig.tight_layout(); fig.savefig(FIG/"Figure_1_corrected_pooled_transformations.png",dpi=600,bbox_inches="tight"); plt.close(fig); d.to_csv(CSV/"Figure_1_source_data.csv",index=False)
    # Main Figure 2: components across n on raw scale.
    r=sim[sim.Transformation=="Raw"]
    fig,axs=plt.subplots(2,2,figsize=(10,7))
    for ax,(name,g) in zip(axs.flat,r.groupby("Scenario",sort=False)):
      ax.plot(g.N_per_group,g.ESI,"o-",label="ESI",color="#D55E00"); ax.set_xscale("log"); ax.set_title(name); ax.set_xlabel("n per group"); ax.set_ylabel("ESI")
    fig.tight_layout(); fig.savefig(FIG/"Figure_2_sample_size.png",dpi=600,bbox_inches="tight"); plt.close(fig); r.to_csv(CSV/"Figure_2_source_data.csv",index=False)
    # Main Figure 3: two empirical use cases automatically chosen by delta extremes.
    benefit=wide.sort_values("Delta_ESI",ascending=False).iloc[0]; neutral=wide.iloc[(wide.Delta_ESI.abs()).argmin()]
    chosen=[benefit,neutral]; fig,axs=plt.subplots(2,2,figsize=(10,7))
    spec_map={(z[2],z[4]):z for z in specs_global}
    frows=[]
    for row_i,row in enumerate(chosen):
      z=spec_map[(row.Dataset,row.Variable)]; _,_,_,df,var,gcol,ga,gb,la,lb=z; sub=df[[var,gcol]].dropna(); x=sub[sub[gcol]==ga][var].to_numpy(); y=sub[sub[gcol]==gb][var].to_numpy(); tx,ty=pooled_transform(x,y,"Rank-normal + min-max")
      for col,(scale,a,b) in enumerate([("Raw",x,y),("Rank-normal + min-max",tx,ty)]):
       ax=axs[row_i,col]; ax.boxplot([a,b],tick_labels=[la,lb],showfliers=False); jitter=np.random.default_rng(7).normal(0,.04,len(a)); ax.scatter(1+jitter,a,s=5,alpha=.25,color="#0072B2"); jitter=np.random.default_rng(8).normal(0,.04,len(b)); ax.scatter(2+jitter,b,s=5,alpha=.25,color="#D55E00"); ax.set_title(f"{row.Dataset}: {var}\n{scale}");
       for group,label,arr in [("A",la,a),("B",lb,b)]:
        for val in arr: frows.append({"Dataset":row.Dataset,"Variable":var,"Scale":scale,"Group":group,"Group_label":label,"Value":val})
    fig.tight_layout(); fig.savefig(FIG/"Figure_3_practical_use_cases.png",dpi=600,bbox_inches="tight"); plt.close(fig); pd.DataFrame(frows).to_csv(CSV/"Figure_3_source_data.csv",index=False)
    # Supplementary Figure S1: explicit component decomposition across n.
    fig,axs=plt.subplots(2,2,figsize=(10,7))
    for ax,metric,label in zip(axs.flat,["Median_abs_d","SD_d","Mean_OVL","ESI"],[r"Median $|d|$",r"SD($d$)","Mean OVL","ESI"]):
     for name,g in r.groupby("Scenario",sort=False): ax.plot(g.N_per_group,g[metric],"o-",label=name)
     ax.set_xscale("log"); ax.set_xlabel("n per group"); ax.set_ylabel(label)
    axs[0,0].legend(fontsize=7); fig.tight_layout(); fig.savefig(FIG/"Figure_S1_component_decomposition.png",dpi=600,bbox_inches="tight"); plt.close(fig); r.to_csv(CSV/"Figure_S1_component_decomposition_source_data.csv",index=False)
    # Supplementary Figure S2: empirical calibration over effect and n under normal shifts.
    cal=[]
    for n in [10,20,50,100,200,500,2000]:
     for effect in [0,.1,.2,.3,.5,.8,1,1.2,1.5]:
      g=[]
      for rr in range(500):
       rg=np.random.default_rng(MASTER_SEED+n*10000+int(effect*1000)+rr); x=rg.normal(0,1,n); y=rg.normal(effect,1,n); d=cohens_d(x,y); g.append((d,abs(d),overlap(x,y)))
      q=pd.DataFrame(g,columns=["Cohens_d","Abs_d","OVL"]); sd=q.Cohens_d.std(ddof=1); cal.append({"N_per_group":n,"Population_d":effect,"ESI":q.Abs_d.median()/sd*(1-q.OVL.mean())})
    cal=pd.DataFrame(cal); pv=cal.pivot(index="Population_d",columns="N_per_group",values="ESI")
    fig,ax=plt.subplots(figsize=(8,5)); im=ax.imshow(pv.values,aspect="auto",origin="lower",cmap="viridis"); ax.set_xticks(range(len(pv.columns)),pv.columns); ax.set_yticks(range(len(pv.index)),pv.index); ax.set_xlabel("n per group"); ax.set_ylabel("Population Cohen's d"); fig.colorbar(im,ax=ax,label="ESI"); fig.tight_layout(); fig.savefig(FIG/"Figure_S2_simulation_calibration.png",dpi=600,bbox_inches="tight"); plt.close(fig); cal.to_csv(CSV/"Figure_S2_simulation_calibration_source_data.csv",index=False)
    # Supplementary Figure S3: kernel and bandwidth sensitivity.
    fig,ax=plt.subplots(figsize=(7,4.5))
    for name,g in kernel.groupby("Kernel",sort=False): ax.plot(g.Bandwidth_multiplier,g.ESI,"o-",label=name)
    ax.set_xlabel("Bandwidth multiplier"); ax.set_ylabel("ESI"); ax.legend(ncol=2,fontsize=8); fig.tight_layout(); fig.savefig(FIG/"Figure_S3_kernel_bandwidth_sensitivity.png",dpi=600,bbox_inches="tight"); plt.close(fig); kernel.to_csv(CSV/"Figure_S3_kernel_bandwidth_sensitivity_source_data.csv",index=False)
    # Supplementary Figure S4: stability estimator sensitivity.
    fig,ax=plt.subplots(figsize=(6,4)); ax.bar(stability.Stability_estimator,stability.ESI,color="#0072B2"); ax.set_ylabel("ESI"); ax.tick_params(axis="x",rotation=20); fig.tight_layout(); fig.savefig(FIG/"Figure_S4_stability_sensitivity.png",dpi=600,bbox_inches="tight"); plt.close(fig); stability.to_csv(CSV/"Figure_S4_stability_sensitivity_source_data.csv",index=False)
    return pd.DataFrame(chosen)

def main():
    for p in (OUT,DATA,FIG,CSV): p.mkdir(parents=True,exist_ok=True)
    global specs_global
    specs_global,overview,manifest=load_datasets(); overview.to_csv(CSV/"Table_S1_dataset_overview.csv",index=False); manifest.to_csv(CSV/"Dataset_manifest.csv",index=False)
    if (CSV/"Table_1_summary.csv").exists():
        full=pd.read_csv(CSV/"Table_1_full_components.csv"); wide=pd.read_csv(CSV/"Table_1_summary.csv")
    else:
        full,wide=real_analysis(specs_global); full.to_csv(CSV/"Table_1_full_components.csv",index=False); wide.to_csv(CSV/"Table_1_summary.csv",index=False)
    if (CSV/"Simulation_summary.csv").exists():
        sim=pd.read_csv(CSV/"Simulation_summary.csv")
    else:
        rep,sim=simulation_analysis(); sim.to_csv(CSV/"Simulation_summary.csv",index=False); rep.to_csv(CSV/"Simulation_replicates.csv",index=False)
    kernel,stability=sensitivity_examples(specs_global); chosen=make_figures(sim,wide,full,kernel,stability); chosen.to_csv(CSV/"Selected_use_cases.csv",index=False)
    meta={"master_seed":MASTER_SEED,"bootstrap_replicates":N_BOOTSTRAPS,"monte_carlo_replicates":N_MONTE_CARLO,"kde_grid_size":KDE_GRID_SIZE,"python":sys.version,"platform":platform.platform(),"numpy":np.__version__,"pandas":pd.__version__,"scipy":scipy.__version__,"sklearn":sklearn.__version__,"matplotlib":mpl.__version__}
    (OUT/"analysis_metadata.json").write_text(json.dumps(meta,indent=2)); print(OUT.resolve())

if __name__=="__main__": main()


Writing esi_pipeline.py


## 2. Run simulations, real-data analyses and supplementary figures


In [ ]:
%run esi_pipeline.py


/content/outputs/esi_submission


<Figure size 1024x768 with 0 Axes>

## 3. Write the publication-quality main-figure generator


In [ ]:
%%writefile generate_publication_figures.py
"""Generate publication-quality ESI Figures 1–3 from final corrected outputs."""
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import esi_pipeline as ep

ROOT=Path("outputs/esi_submission"); SRC=ROOT/"source_csv"; OUT=ROOT/"publication_figures"; OUT.mkdir(exist_ok=True)
BLUE="#1976B6"; ORANGE="#F57C00"; GREEN="#2CA02C"; MAGENTA="#B7195A"; GREY="#7F7F7F"
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":10,"axes.titlesize":12,"axes.labelsize":11,"axes.spines.top":False,"axes.spines.right":False,"figure.dpi":180})

def panel(ax,label): ax.text(-.11,1.04,label,transform=ax.transAxes,fontweight="bold",fontsize=14,va="bottom")
def save(fig,name):
    fig.savefig(OUT/f"{name}.png",dpi=600,bbox_inches="tight",facecolor="white")
    fig.savefig(OUT/f"{name}.pdf",bbox_inches="tight",facecolor="white")
    plt.close(fig)

sim=pd.read_csv(SRC/"Simulation_summary.csv")

# Figure 1 — preserve the earlier multi-metric narrative with corrected ESI.
raw=sim[sim.Transformation.eq("Raw")].copy()
order=["Normal clear","Normal overlap","Skewed lognormal","Exponential vs gamma"]
titles=["Normal clear","Normal overlap","Skewed (lognormal vs lognormal)","Exponential vs gamma"]
fig,axs=plt.subplots(2,2,figsize=(11.5,8.5),constrained_layout=True)
handles=None
for i,(ax,name,title) in enumerate(zip(axs.flat,order,titles)):
    g=raw[raw.Scenario.eq(name)].sort_values("N_per_group"); ax2=ax.twinx()
    p=np.maximum(g.Median_P.to_numpy(),np.nextafter(0,1)); n=g.N_per_group
    l1=ax.plot(n,p,"o-",color=BLUE,lw=1.8,ms=5,label="Median p-value")
    l2=ax2.plot(n,g.Median_abs_d,"s-",color=GREEN,lw=1.6,ms=5,label=r"Median $|d|$")
    l3=ax2.plot(n,g.ESI,"^--",color=ORANGE,lw=1.7,ms=6,label="ESI")
    ax.set_xscale("log"); ax.set_yscale("log"); ax.grid(True,which="both",color="#D8D8D8",lw=.6,alpha=.7)
    ax.set_title(title,pad=7); ax.set_ylabel("p-value (log scale)"); ax2.set_ylabel(r"Effect magnitude / ESI")
    if i>=2: ax.set_xlabel("Sample size per group (n)")
    panel(ax,"abcd"[i]); handles=l1+l2+l3
axs[1,1].legend(handles,[h.get_label() for h in handles],loc="lower left",frameon=True,fontsize=8)
fig.suptitle("p-value, effect magnitude and ESI across sample size (raw scale)",fontsize=15,y=1.02)
save(fig,"Figure_1_publication")

# Figure 2 — corrected heatmap and components for each ESI-preferred scale at n=200.
n200=sim[sim.N_per_group.eq(200)].copy(); base=n200[n200.Transformation.eq("Raw")].set_index("Scenario")
n200["Delta_ESI"]=[r.ESI-base.loc[r.Scenario,"ESI"] for _,r in n200.iterrows()]
trans=["Pooled z-score","Pooled min-max","Log + z-score","Log + min-max","Box-Cox + z-score","Box-Cox + min-max","Rank-normal + min-max"]
short=["Z","Min–max","Log+Z","Log+MM","BC+Z","BC+MM","RankN+MM"]
mat=n200.pivot(index="Scenario",columns="Transformation",values="Delta_ESI").reindex(index=order,columns=trans)
preferred=n200.loc[n200.groupby("Scenario").ESI.idxmax()].set_index("Scenario").reindex(order)
fig=plt.figure(figsize=(13,8.3)); gs=fig.add_gridspec(2,3,height_ratios=[1.15,1],hspace=.68,wspace=.88)
ax=fig.add_subplot(gs[0,:]); vmax=max(abs(mat.to_numpy().min()),abs(mat.to_numpy().max())); im=ax.imshow(mat,cmap="RdBu_r",vmin=-vmax,vmax=vmax,aspect="auto")
ax.set_xticks(range(7),short); ax.set_yticks(range(4),["Normal clear","Normal overlap","Skewed","Exponential vs gamma"]); ax.set_xlabel("Transformation fitted to pooled observations"); ax.set_ylabel("Simulation scenario")
for y in range(4):
 for x in range(7):
  v=mat.iloc[y,x]; ax.text(x,y,f"{v:+.3f}",ha="center",va="center",fontsize=9,fontweight="bold" if abs(v)>=.05 else "normal",color="white" if abs(v)>.55*vmax else "#222")
fig.colorbar(im,ax=ax,fraction=.025,pad=.018,label=r"Absolute change, $\Delta$ESI"); ax.set_title("Transformation effects are scenario-dependent",fontweight="bold",loc="left",pad=10); panel(ax,"a")
metrics=[("Median_abs_d",r"Median $|d|$","Effect magnitude"),("SD_d",r"SD of estimated $d$","Estimation stability (lower is better)"),("Mean_OVL",r"Distributional separation, $1-\mathrm{OVL}$","Distributional separation (higher is better)")]
for j,(col,ylab,title) in enumerate(metrics):
 ax=fig.add_subplot(gs[1,j]); xs=np.arange(4); rv=base.reindex(order)[col].to_numpy(); tv=preferred[col].to_numpy()
 if col=="Mean_OVL": rv,tv=1-rv,1-tv
 for x,a,b in zip(xs,rv,tv): ax.plot([x,x],[a,b],color="#BDBDBD",lw=1.5,zorder=1)
 ax.scatter(xs,rv,s=42,color=GREY,label="Raw",zorder=2); material=np.abs(preferred.ESI.to_numpy()-base.reindex(order).ESI.to_numpy())>=.05
 ax.scatter(xs[material],tv[material],s=48,color=MAGENTA,label="ESI-preferred",zorder=3); ax.scatter(xs[~material],tv[~material],s=58,facecolor="white",edgecolor=MAGENTA,lw=1.7,zorder=3)
 ax.set_xticks(xs,["Normal clear","Normal overlap","Skewed","Exponential vs gamma"],rotation=25,ha="right"); ax.set_ylabel(ylab); ax.set_title(title,fontweight="bold",loc="left",fontsize=11,pad=7); ax.grid(axis="y",color="#DEDEDE",lw=.6); panel(ax,"bcd"[j])
 if j==0: ax.legend(frameon=False,fontsize=8,loc="best")
fig.suptitle("Pooled transformations alter ESI through distinct components",fontweight="bold",fontsize=16,y=.99)
save(fig,"Figure_2_publication")
mat.reset_index().to_csv(SRC/"Figure_2_publication_source_data.csv",index=False)

# Figure 3 — three honest cases: increase, modest increase, decrease.
specs,_,_=ep.load_datasets(); specmap={(z[2],z[4]):z for z in specs}; full=pd.read_csv(SRC/"Table_1_full_components.csv")
cases=[("Breast Cancer Wisconsin","worst area","Breast cancer — worst area"),("Pima Diabetes","Glucose","Pima diabetes — glucose"),("Fetal Health","accelerations","Fetal health — accelerations")]
fig,axs=plt.subplots(3,2,figsize=(11.5,12),constrained_layout=True); colors=[BLUE,"#D95F02"]
source=[]
def violin_box(ax,a,b,labels,ylabel):
 parts=ax.violinplot([a,b],positions=[1,2],showextrema=False,widths=.78)
 for body,c in zip(parts["bodies"],colors): body.set_facecolor(c); body.set_edgecolor(c); body.set_alpha(.32)
 bp=ax.boxplot([a,b],positions=[1,2],widths=.18,showfliers=False,patch_artist=True,medianprops={"color":"black","lw":1.7},boxprops={"facecolor":"white","lw":1.5},whiskerprops={"color":"#777"},capprops={"color":"#777"})
 rng=np.random.default_rng(44)
 for x,v,c in [(1,a,colors[0]),(2,b,colors[1])]: ax.scatter(rng.normal(x,.035,len(v)),v,s=8,color=c,alpha=.28,edgecolors="none")
 ax.set_xticks([1,2],labels); ax.set_ylabel(ylabel); ax.grid(axis="y",color="#E0E0E0",lw=.6)
for row,(ds,var,title) in enumerate(cases):
 z=specmap[(ds,var)]; _,_,_,df,v,gcol,ga,gb,la,lb=z; q=df[[v,gcol]].copy(); q[v]=pd.to_numeric(q[v],errors="coerce"); q=q.dropna(); a=q[q[gcol].eq(ga)][v].to_numpy(); b=q[q[gcol].eq(gb)][v].to_numpy(); ta,tb=ep.pooled_transform(a,b,"Rank-normal + min-max")
 rawm=full[(full.Dataset.eq(ds))&(full.Variable.eq(var))&(full.Scale.eq("Raw"))].iloc[0]; trm=full[(full.Dataset.eq(ds))&(full.Variable.eq(var))&(full.Scale.eq("Rank-normal + min-max"))].iloc[0]; delta=trm.ESI-rawm.ESI
 unit={"worst area":"Worst area","Glucose":r"Glucose (mg dl$^{-1}$)","accelerations":"Accelerations"}[var]
 for col,(scale,x,y,m) in enumerate([("Raw measurement scale",a,b,rawm),("Pooled rank-normal + min–max",ta,tb,trm)]):
  ax=axs[row,col]; violin_box(ax,x,y,[la.title(),lb.title()],unit if col==0 else "Transformed value"); ax.set_title(scale,fontweight="bold",loc="left"); panel(ax,"abcdef"[row*2+col]);
  txt=rf"$d={m.Cohens_d:.2f}$"+"\n"+rf"$p={m.P_value:.1e}$"+"\n"+f"ESI = {m.ESI:.2f}"+("\n"+rf"$\Delta$ESI = {delta:+.2f}" if col else "")
  ax.text(.97,.96,txt,transform=ax.transAxes,ha="right",va="top",bbox={"boxstyle":"round,pad=.3","fc":"white","ec":"#D0D0D0","alpha":.94},fontsize=9)
  for group,label,arr in [("A",la,x),("B",lb,y)]: source.extend({"Dataset":ds,"Variable":var,"Scale":scale,"Group":group,"Group_label":label,"Value":float(val)} for val in arr)
 axs[row,0].text(0,1.17,rf"{title}: $\Delta$ESI = {delta:+.2f}",transform=axs[row,0].transAxes,fontweight="bold",fontsize=12)
fig.suptitle("Pooled rank-normal transformation has variable-dependent effects on ESI",fontweight="bold",fontsize=16,y=1.015)
save(fig,"Figure_3_publication"); pd.DataFrame(source).to_csv(SRC/"Figure_3_publication_source_data.csv",index=False)
print(OUT.resolve())


Writing generate_publication_figures.py


## 4. Generate final Figures 1–3 as 600-dpi PNG and vector PDF


In [ ]:
%run generate_publication_figures.py


## 5. Create the formatted journal source-data Excel workbook


In [ ]:
from pathlib import Path
import pandas as pd, json

ROOT=Path("outputs/esi_submission")
SRC=ROOT/"source_csv"
XLSX=ROOT/"Source_Data_Colab.xlsx"

sheet_files={
 "Table 1":"Table_1_summary.csv",
 "Dataset Overview":"Table_S1_dataset_overview.csv",
 "Dataset Manifest":"Dataset_manifest.csv",
 "Figure 1":"Figure_1_source_data.csv",
 "Figure 2":"Figure_2_publication_source_data.csv",
 "Figure 3":"Figure_3_publication_source_data.csv",
 "Supp Figure S1":"Figure_S1_component_decomposition_source_data.csv",
 "Supp Figure S2":"Figure_S2_simulation_calibration_source_data.csv",
 "Supp Figure S3":"Figure_S3_kernel_bandwidth_sensitivity_source_data.csv",
 "Supp Figure S4":"Figure_S4_stability_sensitivity_source_data.csv",
 "Simulation Summary":"Simulation_summary.csv",
 "ESI Components":"Table_1_full_components.csv",
}

with pd.ExcelWriter(XLSX,engine="xlsxwriter") as writer:
    wb=writer.book
    title=wb.add_format({"bold":True,"font_color":"white","bg_color":"#17365D","font_size":15,"align":"left","valign":"vcenter"})
    header=wb.add_format({"bold":True,"font_color":"#17365D","bg_color":"#D9EAF7","border":0,"bottom":2,"bottom_color":"#17365D","text_wrap":True,"valign":"top"})
    text=wb.add_format({"valign":"top","text_wrap":True})
    integer=wb.add_format({"num_format":"#,##0","valign":"top"})
    decimal=wb.add_format({"num_format":"0.0000","valign":"top"})
    pvalue=wb.add_format({"num_format":"0.00E+00","valign":"top"})
    note_label=wb.add_format({"bold":True,"font_color":"#17365D","bg_color":"#D9EAF7","valign":"top"})
    note_value=wb.add_format({"bg_color":"#F3F6F9","text_wrap":True,"valign":"top"})
    readme=wb.add_worksheet("README"); writer.sheets["README"]=readme
    readme.merge_range("A1:H1","Effect Separation Index — Journal Source Data",title); readme.set_row(0,28)
    notes=[
      ("Purpose","Numerical source data for all corrected main and supplementary figures and tables."),
      ("Primary specification","Pooled transformations; 2,000 within-group bootstraps; 500 paired Monte Carlo replicates; Gaussian KDE with robust Silverman bandwidth."),
      ("Interpretation","ESI is sample-dependent and is not an intrinsic effect size, clinical-importance measure, or predictive-performance metric."),
      ("Transformation comparison","Delta_ESI = ESI(transformed) - ESI(raw)."),
      ("Dataset correction","Heart Disease Prediction (918 records) and UCI Heart Failure Clinical Records (299 records) are distinct datasets."),
      ("Reproduction","Run every Colab cell in order. The final cell creates ESI_Colab_complete_outputs.zip."),
    ]
    for i,(a,b) in enumerate(notes,2): readme.write(i,0,a,note_label); readme.write(i,1,b,note_value)
    readme.set_column("A:A",24); readme.set_column("B:B",95); readme.hide_gridlines(2)

    for sheet,filename in sheet_files.items():
        path=SRC/filename
        if not path.exists(): raise FileNotFoundError(f"Missing required source data: {path}")
        df=pd.read_csv(path)
        df.to_excel(writer,sheet_name=sheet,index=False,startrow=2,header=False)
        ws=writer.sheets[sheet]; ws.hide_gridlines(2); ws.freeze_panes(3,0)
        ws.merge_range(0,0,0,max(len(df.columns)-1,0),f"{sheet} source data",title); ws.set_row(0,26)
        for c,name in enumerate(df.columns): ws.write(2,c,name,header)
        for c,name in enumerate(df.columns):
            vals=df[name].astype(str).head(250); width=max(len(str(name)),*(len(v) for v in vals)) if len(vals) else len(str(name))
            width=min(55,max(11,width+2)); fmt=text
            if any(k in name for k in ["P_value","Median_P"]): fmt=pvalue
            elif any(k in name for k in ["ESI","OVL","Cohens","Abs_d","SD_","Delta","Denominator"]): fmt=decimal
            elif any(k in name for k in ["N_A","N_B","Rows","Columns","bootstraps","Replicate"]): fmt=integer
            ws.set_column(c,c,width,fmt)
        ws.autofilter(2,0,len(df)+2,len(df.columns)-1)

assert XLSX.exists() and XLSX.stat().st_size>10000
print(f"Created: {XLSX} ({XLSX.stat().st_size/1e6:.2f} MB)")


Created: outputs/esi_submission/Source_Data_Colab.xlsx (0.20 MB)


## 6. Validate every required output, package everything and download


In [ ]:
from pathlib import Path
import json, zipfile

ROOT=Path("outputs/esi_submission")
required=[
 ROOT/"Source_Data_Colab.xlsx",
 ROOT/"source_csv"/"Simulation_summary.csv",
 ROOT/"source_csv"/"Table_1_summary.csv",
 ROOT/"publication_figures"/"Figure_1_publication.png",
 ROOT/"publication_figures"/"Figure_2_publication.png",
 ROOT/"publication_figures"/"Figure_3_publication.png",
]
for p in required:
    assert p.exists() and p.stat().st_size>0, f"Missing output: {p}"

sim=__import__("pandas").read_csv(ROOT/"source_csv"/"Simulation_summary.csv")
assert "SD_d" in sim.columns and "SD_abs_d" not in sim.columns
real=__import__("pandas").read_csv(ROOT/"source_csv"/"Table_1_full_components.csv")
assert len(real)==86 and set(real["Scale"])=={"Raw","Rank-normal + min-max"}

archive=Path("ESI_Colab_complete_outputs.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.rglob("*"):
        if p.is_file() and "downloaded_data" not in p.parts:
            z.write(p,p.as_posix())
    z.write("esi_pipeline.py","esi_pipeline.py")
    z.write("generate_publication_figures.py","generate_publication_figures.py")
print(f"Validated and packaged: {archive} ({archive.stat().st_size/1e6:.2f} MB)")

try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print("Not running in Colab; download the ZIP from the file browser.")
